# Hash Ensemble Ablations on CIFAR-10

Companion notebook to `NOTES.md`. Tests whether ensembles of HashedNets — especially with **heterogeneous codebook sizes** — beat single HashedNets at matched parameter budget, and whether smart aggregation buys anything beyond uniform averaging.

**Hypotheses tested (see NOTES.md for full statements):**
- H1: ensemble of N hashed nets at K beats single hashed net at N·K
- H2: heterogeneous K (different per model) beats homogeneous K at matched total budget
- H3-H4: confidence-weighted / trimmed aggregation beats plain mean
- H5: joint training beats independent training
- H6: ensemble variance across hash-seeds is much smaller than single
- H7: layer-wise heterogeneous K adds further diversity gain

**Default runtime on Colab T4: ~100 minutes.** Set `FAST_MODE = True` for a ~25 min smoke pass that still touches all ablations.

In [ ]:
import os, math, time, json, random
from tqdm.auto import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt

# -------- global config --------
FAST_MODE = False  # True = quick smoke (~30 min), False = full (~110 min)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = './data'
RESULTS_PATH = './results.json'

BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 5e-4
EPOCHS = 8 if FAST_MODE else 15
EPOCHS_JOINT = 6 if FAST_MODE else 10

# DSCNN architecture (matches the spirit of hash_kws_lab/models.py, with extra hash-aware bits)
CHANNELS = 64
NUM_BLOCKS = 3
NUM_CLASSES = 10
USE_RESIDUAL = True
SIGNED_HASH = True
USE_SE = True            # squeeze-and-excitation per block (hashed FCs inside)
SE_REDUCTION = 8         # bottleneck ratio in SE
HASH_BRANCHES = 1        # parallel hash branches per hashed layer; 1 = standard HashedNet, 2 = double-hash
HASH_AWARE_INIT = True   # scale codebook init by 1/sqrt(fan_in_equiv) so virtual W variance matches Kaiming

# Ensemble + sweep
ENSEMBLE_N = 3 if FAST_MODE else 5
DISPERSION_N = 4 if FAST_MODE else 6
K_SWEEP = [256, 1024, 4096] if FAST_MODE else [128, 256, 512, 1024, 2048, 4096]
K_HOMO = 1024  # baseline per-layer codebook size for ensemble experiments

# Heterogeneous K: sum(K_HETERO) = ENSEMBLE_N * K_HOMO  (matched total budget vs A2)
HETERO_RATIOS = {
    3: [0.5, 1.0, 1.5],                 # sum = 3
    5: [0.25, 0.5, 1.0, 1.25, 2.0],     # sum = 5
}
K_HETERO = [int(K_HOMO * r) for r in HETERO_RATIOS[ENSEMBLE_N]]
assert abs(sum(K_HETERO) - ENSEMBLE_N * K_HOMO) <= ENSEMBLE_N

print(f'Device: {DEVICE}  FAST_MODE: {FAST_MODE}')
print(f'Architecture: DSCNN channels={CHANNELS} num_blocks={NUM_BLOCKS} residual={USE_RESIDUAL} '
      f'signed_hash={SIGNED_HASH} se={USE_SE}(r={SE_REDUCTION}) branches={HASH_BRANCHES} aware_init={HASH_AWARE_INIT}')
print(f'epochs={EPOCHS}  ensemble_N={ENSEMBLE_N}  K_HOMO={K_HOMO}')
print(f'K sweep: {K_SWEEP}')
print(f'K hetero: {K_HETERO}  (sum={sum(K_HETERO)}, target={ENSEMBLE_N*K_HOMO})')

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_all_seeds(0)

## Data

CIFAR-10 with light augmentation. Test loader without augmentation. Also a corrupted test loader for the OOD-lite ablation (A8).

In [ ]:
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)

tf_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
tf_test = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

trainset = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=tf_train)
testset = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=tf_test)

trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
testloader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# OOD-lite: gaussian noise + brightness shift on the test set, on the fly
class CorruptedCIFAR(torch.utils.data.Dataset):
    def __init__(self, base, noise_std=0.15, bright=0.2):
        self.base = base; self.noise_std = noise_std; self.bright = bright
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        x = x + self.bright + torch.randn_like(x) * self.noise_std
        return x, y

ood_loader = DataLoader(CorruptedCIFAR(testset), batch_size=256, shuffle=False, num_workers=2)
print(f'train: {len(trainset)}, test: {len(testset)}')

## Architecture

DSCNN with **all** trainable layers hashed: stem `Conv2d` (3→C, 3×3, stride 2), `num_blocks` × (depthwise 3×3 + pointwise 1×1) blocks with optional residual, global average pool, hashed final `Linear`. Mirrors `hash_kws_lab/models.py` from the diploma codebase, just adapted to CIFAR-10 input (3-channel 32×32) instead of mel-features.

Hash is analytic (multiplicative residue with large primes), takes a per-layer `layer_id` plus a per-model `hash_seed` so different ensemble members get statistically independent collision patterns. Sign function via second analytic hash mod 2 — kills the systematic bias to the codebook mean. Per-layer `K` is configurable: the ensemble experiments use a single K shared across layers in a given member; the layer-wise ablation A6 varies K per layer.

In [ ]:
# Hash primes (analytic hash, same spirit as hash_kws_lab/models.py)
P_OC, P_IC, P_KH, P_KW, P_LAYER = 1337, 7919, 2971, 6151, 104729
P_SEED = 31337
S_A, S_B, S_C, S_SEED = 4099, 6151, 14887, 67867
# extra primes for second branch (HASH_BRANCHES >= 2): different mixing constants
P_OC2, P_IC2, P_KH2, P_KW2, P_LAYER2, P_SEED2 = 9311, 4877, 8111, 3613, 70663, 99991
S_A2, S_B2, S_C2, S_SEED2 = 5527, 7187, 11789, 50069


def _bits_to_sign(bits):
    return bits.to(torch.float32).mul_(2.0).sub_(1.0)


def _aware_std(fan_in_equiv):
    """Codebook init std so Var(W_eff) ≈ Var(Kaiming_dense) regardless of K. Used iff HASH_AWARE_INIT."""
    return 1.0 / math.sqrt(max(fan_in_equiv, 1))


class HashedLinear(nn.Module):
    """Linear layer with K-codebook + analytic hash + analytic sign. Optional N-branch double-hashing."""
    def __init__(self, in_dim, out_dim, K, layer_id, hash_seed=0,
                 signed=True, branches=1, aware_init=True):
        super().__init__()
        self.K = K; self.signed = signed; self.branches = branches
        std = _aware_std(in_dim) if aware_init else 0.01
        self.codebook = nn.Parameter(torch.randn(K) * std)
        self.bias = nn.Parameter(torch.zeros(out_dim))
        i = torch.arange(out_dim).view(-1, 1)
        j = torch.arange(in_dim).view(1, -1)
        for b in range(branches):
            if b == 0:
                h = (i * P_OC + j * P_IC + layer_id * P_LAYER + hash_seed * P_SEED) % K
                s = _bits_to_sign((i * S_A + j * S_B + layer_id * S_C + hash_seed * S_SEED) % 2)
            else:
                h = (i * P_OC2 + j * P_IC2 + (layer_id + 7) * P_LAYER2 + hash_seed * P_SEED2) % K
                s = _bits_to_sign((i * S_A2 + j * S_B2 + (layer_id + 7) * S_C2 + hash_seed * S_SEED2) % 2)
            self.register_buffer(f'hash_idx_{b}', h, persistent=False)
            self.register_buffer(f'signs_{b}', s, persistent=False)

    def materialize(self):
        W = 0
        for b in range(self.branches):
            Wb = self.codebook[getattr(self, f'hash_idx_{b}')]
            if self.signed: Wb = Wb * getattr(self, f'signs_{b}')
            W = Wb if isinstance(W, int) else W + Wb
        if self.branches > 1: W = W / math.sqrt(self.branches)
        return W

    def forward(self, x): return F.linear(x, self.materialize(), self.bias)
    def virtual_params(self): return self.hash_idx_0.numel()
    def real_params(self): return self.K + self.bias.numel()


class HashedConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, K, layer_id, hash_seed=0,
                 stride=1, padding=0, signed=True, branches=1, aware_init=True):
        super().__init__()
        self.K = K; self.signed = signed; self.branches = branches
        self.stride = stride; self.padding = padding
        kh, kw = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kh, self.kw = kh, kw
        fan_in_eq = in_channels * kh * kw
        std = _aware_std(fan_in_eq) if aware_init else 0.01
        self.codebook = nn.Parameter(torch.randn(K) * std)
        self.bias = nn.Parameter(torch.zeros(out_channels))
        oc = torch.arange(out_channels).view(-1, 1, 1, 1)
        ic = torch.arange(in_channels).view(1, -1, 1, 1)
        ih = torch.arange(kh).view(1, 1, -1, 1)
        iw = torch.arange(kw).view(1, 1, 1, -1)
        for b in range(branches):
            if b == 0:
                h = (oc*P_OC + ic*P_IC + ih*P_KH + iw*P_KW + layer_id*P_LAYER + hash_seed*P_SEED) % K
                s = _bits_to_sign((oc*S_A + ic*S_B + ih*S_C + iw*(S_A+S_B) + layer_id*(S_C+11) + hash_seed*S_SEED) % 2)
            else:
                h = (oc*P_OC2 + ic*P_IC2 + ih*P_KH2 + iw*P_KW2 + (layer_id+7)*P_LAYER2 + hash_seed*P_SEED2) % K
                s = _bits_to_sign((oc*S_A2 + ic*S_B2 + ih*S_C2 + iw*(S_A2+S_B2) + (layer_id+7)*(S_C2+13) + hash_seed*S_SEED2) % 2)
            self.register_buffer(f'hash_idx_{b}', h, persistent=False)
            self.register_buffer(f'signs_{b}', s, persistent=False)

    def materialize(self):
        W = 0
        for b in range(self.branches):
            Wb = self.codebook[getattr(self, f'hash_idx_{b}')]
            if self.signed: Wb = Wb * getattr(self, f'signs_{b}')
            W = Wb if isinstance(W, int) else W + Wb
        if self.branches > 1: W = W / math.sqrt(self.branches)
        return W

    def forward(self, x):
        return F.conv2d(x, self.materialize(), self.bias, stride=self.stride, padding=self.padding)

    def virtual_params(self): return self.hash_idx_0.numel()
    def real_params(self): return self.K + self.bias.numel()


class HashedDepthwiseConv2d(nn.Module):
    def __init__(self, channels, kernel_size, K, layer_id, hash_seed=0,
                 stride=1, padding=0, signed=True, branches=1, aware_init=True):
        super().__init__()
        self.K = K; self.signed = signed; self.branches = branches
        self.channels = channels
        self.stride = stride; self.padding = padding
        kh, kw = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kh, self.kw = kh, kw
        fan_in_eq = kh * kw
        std = _aware_std(fan_in_eq) if aware_init else 0.01
        self.codebook = nn.Parameter(torch.randn(K) * std)
        self.bias = nn.Parameter(torch.zeros(channels))
        ch = torch.arange(channels).view(-1, 1, 1)
        ih = torch.arange(kh).view(1, -1, 1)
        iw = torch.arange(kw).view(1, 1, -1)
        for b in range(branches):
            if b == 0:
                h = (ch*P_OC + ih*P_KH + iw*P_KW + layer_id*P_LAYER + hash_seed*P_SEED) % K
                s = _bits_to_sign((ch*S_A + ih*S_B + iw*S_C + layer_id*(S_A+29) + hash_seed*S_SEED) % 2)
            else:
                h = (ch*P_OC2 + ih*P_KH2 + iw*P_KW2 + (layer_id+7)*P_LAYER2 + hash_seed*P_SEED2) % K
                s = _bits_to_sign((ch*S_A2 + ih*S_B2 + iw*S_C2 + (layer_id+7)*(S_A2+31) + hash_seed*S_SEED2) % 2)
            self.register_buffer(f'hash_idx_{b}', h, persistent=False)
            self.register_buffer(f'signs_{b}', s, persistent=False)

    def materialize(self):
        W = 0
        for b in range(self.branches):
            Wb = self.codebook[getattr(self, f'hash_idx_{b}')]
            if self.signed: Wb = Wb * getattr(self, f'signs_{b}')
            W = Wb if isinstance(W, int) else W + Wb
        if self.branches > 1: W = W / math.sqrt(self.branches)
        return W.unsqueeze(1)

    def forward(self, x):
        return F.conv2d(x, self.materialize(), self.bias, stride=self.stride,
                        padding=self.padding, groups=self.channels)

    def virtual_params(self): return self.hash_idx_0.numel()
    def real_params(self): return self.K + self.bias.numel()


class HashedSEBlock(nn.Module):
    """Squeeze-and-excitation. Two hashed FCs inside (squeeze C→C/r, excite C/r→C). Cheap, hashable."""
    def __init__(self, channels, reduction, K_se, layer_id, hash_seed,
                 signed=True, branches=1, aware_init=True):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.fc1 = HashedLinear(channels, hidden, K_se, layer_id, hash_seed,
                                signed=signed, branches=branches, aware_init=aware_init)
        self.fc2 = HashedLinear(hidden, channels, K_se, layer_id + 13, hash_seed,
                                signed=signed, branches=branches, aware_init=aware_init)

    def forward(self, x):
        z = x.mean(dim=(2, 3))
        z = F.relu(self.fc1(z))
        z = torch.sigmoid(self.fc2(z))
        return x * z.unsqueeze(-1).unsqueeze(-1)


class DSCNNBlock(nn.Module):
    def __init__(self, channels, dw, pw, residual=True, se=None):
        super().__init__()
        self.dw = dw; self.bn_dw = nn.BatchNorm2d(channels)
        self.pw = pw; self.bn_pw = nn.BatchNorm2d(channels)
        self.residual = residual
        self.se = se

    def forward(self, x):
        s = x
        x = F.relu(self.bn_dw(self.dw(x)))
        x = self.bn_pw(self.pw(x))
        if self.se is not None: x = self.se(x)
        if self.residual: x = x + s
        return F.relu(x)


# Atomic hashed layer types — used to count parameters without recursing into the whole model.
HASHED_LAYER_TYPES = (HashedLinear, HashedConv2d, HashedDepthwiseConv2d)


class HashedDSCNN(nn.Module):
    """
    DSCNN for CIFAR-10 with hashing on stem conv, depthwise, pointwise, fc, and (optionally) SE FCs.
    K_per_layer:
      int  → same K across all hashed layers
      dict → keys 'stem', 'dw' (list[num_blocks] | int), 'pw' (list | int), 'fc', 'se' (used iff USE_SE).
             None for any entry = leave that layer dense.
    """
    def __init__(self, K_per_layer, hash_seed=0, channels=CHANNELS, num_blocks=NUM_BLOCKS,
                 num_classes=NUM_CLASSES, residual=USE_RESIDUAL, signed=SIGNED_HASH, dropout=0.1,
                 use_se=USE_SE, se_reduction=SE_REDUCTION, branches=HASH_BRANCHES,
                 aware_init=HASH_AWARE_INIT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        if isinstance(K_per_layer, int):
            K_stem = K_per_layer
            K_dw = [K_per_layer] * num_blocks
            K_pw = [K_per_layer] * num_blocks
            K_fc = K_per_layer
            K_se = K_per_layer
        elif isinstance(K_per_layer, dict):
            K_stem = K_per_layer.get('stem', None)
            K_dw = K_per_layer.get('dw', [None] * num_blocks)
            K_pw = K_per_layer.get('pw', [None] * num_blocks)
            K_fc = K_per_layer.get('fc', None)
            K_se = K_per_layer.get('se', None)
            if isinstance(K_dw, int): K_dw = [K_dw] * num_blocks
            if isinstance(K_pw, int): K_pw = [K_pw] * num_blocks
        else:
            raise TypeError

        kw_hash = dict(signed=signed, branches=branches, aware_init=aware_init)
        layer_id = 0

        if K_stem is None:
            self.stem = nn.Conv2d(3, channels, 3, stride=2, padding=1)
        else:
            self.stem = HashedConv2d(3, channels, 3, K_stem, layer_id, hash_seed,
                                     stride=2, padding=1, **kw_hash); layer_id += 1
        self.bn_stem = nn.BatchNorm2d(channels)

        blocks = []
        for b in range(num_blocks):
            kdw = K_dw[b] if b < len(K_dw) else K_dw[-1]
            kpw = K_pw[b] if b < len(K_pw) else K_pw[-1]
            if kdw is None:
                dw = nn.Conv2d(channels, channels, 3, padding=1, groups=channels)
            else:
                dw = HashedDepthwiseConv2d(channels, 3, kdw, layer_id, hash_seed,
                                           padding=1, **kw_hash); layer_id += 1
            if kpw is None:
                pw = nn.Conv2d(channels, channels, 1)
            else:
                pw = HashedConv2d(channels, channels, 1, kpw, layer_id, hash_seed,
                                  **kw_hash); layer_id += 1

            se = None
            if use_se and K_se is not None:
                se = HashedSEBlock(channels, se_reduction, K_se, layer_id, hash_seed, **kw_hash)
                layer_id += 2

            blocks.append(DSCNNBlock(channels, dw, pw, residual=residual, se=se))
        self.blocks = nn.ModuleList(blocks)

        self.pool = nn.AdaptiveAvgPool2d(1)
        if K_fc is None:
            self.fc = nn.Linear(channels, num_classes)
        else:
            self.fc = HashedLinear(channels, num_classes, K_fc, layer_id, hash_seed, **kw_hash)

    def forward(self, x):
        x = F.relu(self.bn_stem(self.stem(x)))
        for blk in self.blocks: x = blk(x)
        x = self.pool(x).view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x)

    def virtual_params(self):
        """Total weight-tensor size if the model were dense — sum over atomic hashed/dense layers only."""
        n = 0
        for m in self.modules():
            if m is self:
                continue
            if isinstance(m, HASHED_LAYER_TYPES):
                n += m.virtual_params()
            elif isinstance(m, (nn.Conv2d, nn.Linear)):
                n += m.weight.numel()
        return n

    def real_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def DenseDSCNN():
    """All-dense baseline (no hashing anywhere)."""
    return HashedDSCNN(K_per_layer={'stem': None, 'dw': [None]*NUM_BLOCKS,
                                    'pw': [None]*NUM_BLOCKS, 'fc': None, 'se': None})


def make_proportional_K(total_budget, model_sample):
    """Helper for proportional K assignment per layer (used by A6). Walks atomic hashed layers only."""
    sizes = []
    layer_names = []
    for name, m in model_sample.named_modules():
        if isinstance(m, HASHED_LAYER_TYPES):
            sizes.append(m.virtual_params())
            layer_names.append(name)
    sizes_arr = np.array(sizes, dtype=np.float64)
    weights = np.sqrt(sizes_arr)
    weights = weights / weights.sum()
    Ks = (weights * total_budget).astype(int).clip(min=8).tolist()
    return dict(zip(layer_names, Ks)), sizes


# sanity
m_d = DenseDSCNN()
m_h = HashedDSCNN(K_per_layer=K_HOMO, hash_seed=1)
print(f'dense:           real={m_d.real_params():,}  virtual={m_d.virtual_params():,}')
print(f'hashed K={K_HOMO}: real={m_h.real_params():,}  virtual={m_h.virtual_params():,}  '
      f'ratio={m_h.virtual_params()/max(m_h.real_params(),1):.1f}x')
del m_d, m_h

## Training utilities

Single-model and joint-ensemble trainers. Both share the same data loader and reporting format. Joint trainer optimizes a single mean-of-softmax NLL across all members simultaneously.

In [ ]:
@torch.no_grad()
def evaluate_logits(model, loader, desc=None):
    """Returns (acc, all_logits[N,10], all_labels[N]). If desc set, shows tqdm."""
    model.eval()
    all_logits, all_labels = [], []
    correct, total = 0, 0
    iterable = tqdm(loader, desc=desc, leave=False) if desc else loader
    for x, y in iterable:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        correct += (logits.argmax(-1) == y).sum().item()
        total += y.size(0)
        all_logits.append(logits.cpu()); all_labels.append(y.cpu())
    return correct / total, torch.cat(all_logits), torch.cat(all_labels)


def train_one(model, epochs=EPOCHS, log_prefix=''):
    """Train one model. Per-batch tqdm with running loss; per-epoch eval."""
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_acc = 0.0
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        ema_loss = None
        pbar = tqdm(trainloader, desc=f'{log_prefix} ep{ep+1}/{epochs}', leave=False)
        for x, y in pbar:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward(); opt.step()
            ema_loss = loss.item() if ema_loss is None else 0.9 * ema_loss + 0.1 * loss.item()
            pbar.set_postfix(loss=f'{ema_loss:.3f}', lr=f'{sched.get_last_lr()[0]:.4f}')
        sched.step()
        acc, _, _ = evaluate_logits(model, testloader)
        best_acc = max(best_acc, acc)
        tqdm.write(f'{log_prefix} ep{ep+1}/{epochs}  loss={ema_loss:.3f}  test_acc={acc:.4f}  ({time.time()-t0:.0f}s)')
    return model, best_acc


def train_joint(models, epochs=EPOCHS_JOINT, log_prefix=''):
    """Train an ensemble end-to-end with mean-of-probs aggregated NLL. Per-batch tqdm."""
    for m in models: m.to(DEVICE)
    params = [p for m in models for p in m.parameters()]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    t0 = time.time()
    for ep in range(epochs):
        for m in models: m.train()
        ema_loss = None
        pbar = tqdm(trainloader, desc=f'{log_prefix} ep{ep+1}/{epochs}', leave=False)
        for x, y in pbar:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            probs = torch.stack([F.softmax(m(x), dim=-1) for m in models])  # [N,B,C]
            avg = probs.mean(0)
            loss = F.nll_loss(torch.log(avg.clamp_min(1e-10)), y)
            loss.backward(); opt.step()
            ema_loss = loss.item() if ema_loss is None else 0.9 * ema_loss + 0.1 * loss.item()
            pbar.set_postfix(loss=f'{ema_loss:.3f}', lr=f'{sched.get_last_lr()[0]:.4f}')
        sched.step()
        # ensemble-level eval each epoch
        for m in models: m.eval()
        with torch.no_grad():
            correct, total = 0, 0
            for x, y in tqdm(testloader, desc=f'{log_prefix} eval ep{ep+1}', leave=False):
                x, y = x.to(DEVICE), y.to(DEVICE)
                avg = torch.stack([F.softmax(m(x), dim=-1) for m in models]).mean(0)
                correct += (avg.argmax(-1) == y).sum().item(); total += y.size(0)
        tqdm.write(f'{log_prefix} ep{ep+1}/{epochs}  loss={ema_loss:.3f}  ens_acc={correct/total:.4f}  ({time.time()-t0:.0f}s)')
    return models


# Persistent results dict — written incrementally
results = {}
def save_results():
    with open(RESULTS_PATH, 'w') as f:
        json.dump(results, f, indent=2)
    tqdm.write(f'  → results saved to {RESULTS_PATH}')

## Aggregators

All aggregators take stacked logits `[N, B, C]` from N models on a batch of B samples and return logits-equivalent or probabilities of shape `[B, C]`. Implementation kept short: the conceptual differences between them are what matter, not the code.

In [ ]:
def agg_mean_logits(logits_NBC):
    return logits_NBC.mean(0)

def agg_mean_probs(logits_NBC):
    return F.softmax(logits_NBC, dim=-1).mean(0)  # already probs

def agg_confidence_weighted(logits_NBC):
    """Weight each model's prediction by inverse entropy of its softmax (per-input)."""
    probs = F.softmax(logits_NBC, dim=-1)            # [N,B,C]
    ent = -(probs * probs.clamp_min(1e-10).log()).sum(-1)   # [N,B]
    w = 1.0 / (ent + 1e-3)                             # [N,B]
    w = w / w.sum(0, keepdim=True)                      # normalize over N
    return (w.unsqueeze(-1) * probs).sum(0)             # [B,C]

def agg_trimmed(logits_NBC, trim=0.2):
    """Trimmed mean of logits across N. Drops top/bottom trim-fraction per (B,C) cell."""
    N = logits_NBC.size(0)
    sorted_, _ = torch.sort(logits_NBC, dim=0)
    k = max(1, int(N * trim))
    if 2 * k >= N: return sorted_.mean(0)
    return sorted_[k:N-k].mean(0)

def agg_majority_vote(logits_NBC):
    """Hard vote, returned as one-hot probs (for accuracy comparison only)."""
    preds = logits_NBC.argmax(-1)            # [N,B]
    onehot = F.one_hot(preds, num_classes=logits_NBC.size(-1)).float()  # [N,B,C]
    return onehot.mean(0)

AGGREGATORS = {
    'mean_logits': agg_mean_logits,
    'mean_probs': agg_mean_probs,
    'conf_weighted': agg_confidence_weighted,
    'trimmed': agg_trimmed,
    'majority_vote': agg_majority_vote,
}

def ensemble_accuracy(per_model_logits, labels, aggregator):
    """per_model_logits: list of tensors [B,C]. Returns scalar accuracy."""
    stacked = torch.stack(per_model_logits)  # [N,B,C]
    out = aggregator(stacked)
    return (out.argmax(-1) == labels).float().mean().item()

## A0. Dense baseline

Standard CNN with no hashing. Used as the upper anchor ("how good can the architecture get with full FC?").

In [ ]:
set_all_seeds(42)
model_dense = DenseDSCNN()
_, acc_dense = train_one(model_dense, log_prefix='[A0 dense]')
results['A0_dense'] = {'acc': acc_dense, 'params': model_dense.real_params(),
                        'virtual': model_dense.virtual_params()}
print(f'A0 dense: acc={acc_dense:.4f}  real={model_dense.real_params():,}  virtual={model_dense.virtual_params():,}')
save_results()

## A1. Single HashedNet sweep over K

Tests the basic relationship between codebook size and accuracy for a single hashed net. The key data point for H1 is: **the accuracy at total budget = N · K_HOMO** (will compare to A2 ensemble at homogeneous K_HOMO with N members).

In [ ]:
results['A1_sweep'] = {}
for K in K_SWEEP:
    set_all_seeds(100 + K)
    m = HashedDSCNN(K_per_layer=K, hash_seed=100 + K)
    _, acc = train_one(m, log_prefix=f'[A1 K={K}]')
    results['A1_sweep'][K] = {'acc': acc, 'real': m.real_params(), 'virtual': m.virtual_params()}
    print(f'A1 K={K}: acc={acc:.4f}  real={m.real_params():,}  virtual={m.virtual_params():,}')
    del m
save_results()

## A2. Same-K ensemble (H1)

Train `ENSEMBLE_N` models at `K = K_HOMO`, all with different hash seeds. Aggregate with `mean_probs`.

**The H1 falsifier:** if A2's ensemble accuracy ≤ A1's accuracy at K = N·K_HOMO, then ensembling buys nothing beyond "have more parameters" — and the architectural angle is empty. Watch this comparison.

In [ ]:
homo_models, homo_logits, single_homo_accs = [], [], []
for k in range(ENSEMBLE_N):
    set_all_seeds(200 + k)
    m = HashedDSCNN(K_per_layer=K_HOMO, hash_seed=200 + k)
    _, acc = train_one(m, log_prefix=f'[A2 m{k} K={K_HOMO}]')
    _, logits, labels = evaluate_logits(m, testloader)
    homo_models.append(m); homo_logits.append(logits); single_homo_accs.append(acc)
    print(f'A2 m{k}: single_acc={acc:.4f}')

ens_acc_homo = ensemble_accuracy(homo_logits, labels, agg_mean_probs)
results['A2_same_K'] = {
    'K': K_HOMO, 'N': ENSEMBLE_N,
    'single_accs': single_homo_accs,
    'best_single': max(single_homo_accs),
    'ensemble_meanp': ens_acc_homo,
    'total_codebook_budget': K_HOMO * ENSEMBLE_N,
}
print(f'A2 ensemble (mean_probs): {ens_acc_homo:.4f}  | best single: {max(single_homo_accs):.4f}')
save_results()

## A3. Different-K ensemble (H2 — the key claim)

Train `ENSEMBLE_N` models with **different** K each. Total codebook budget matched to A2 (`sum(K_HETERO) ≈ N·K_HOMO`).

If A3 > A2 at matched total budget, **heterogeneous K provides diversity that homogeneous K cannot**. This is the signature observation that distinguishes this idea from plain bagging.

In [ ]:
hetero_models, hetero_logits, single_hetero_accs = [], [], []
for k, K in enumerate(K_HETERO):
    set_all_seeds(300 + k)
    m = HashedDSCNN(K_per_layer=K, hash_seed=300 + k)
    _, acc = train_one(m, log_prefix=f'[A3 m{k} K={K}]')
    _, logits, labels = evaluate_logits(m, testloader)
    hetero_models.append(m); hetero_logits.append(logits); single_hetero_accs.append(acc)
    print(f'A3 m{k} K={K}: single_acc={acc:.4f}')

ens_acc_hetero = ensemble_accuracy(hetero_logits, labels, agg_mean_probs)
results['A3_different_K'] = {
    'K_per_model': K_HETERO, 'N': ENSEMBLE_N,
    'single_accs': single_hetero_accs,
    'best_single': max(single_hetero_accs),
    'ensemble_meanp': ens_acc_hetero,
    'total_codebook_budget': sum(K_HETERO),
}
print(f'A3 ensemble (mean_probs): {ens_acc_hetero:.4f}  | best single: {max(single_hetero_accs):.4f}')
print(f'   total codebook budget A3 = {sum(K_HETERO)}, A2 = {K_HOMO * ENSEMBLE_N}')
save_results()

## A4. Aggregation methods comparison (H3, H4)

Reuses the trained models from A3 (heterogeneous K). For each aggregator, compute ensemble accuracy. Also compute on A2 for comparison — does the gap between aggregators differ between homogeneous and heterogeneous settings?

**Diagnostics:** per-input per-model entropy distribution (does heterogeneous-K ensemble produce more variable confidence than homogeneous-K?) and per-input pairwise agreement rate (have models specialized?).

In [ ]:
def aggregator_table(per_model_logits, labels, label_str):
    out = {}
    for name, agg in AGGREGATORS.items():
        a = ensemble_accuracy(per_model_logits, labels, agg)
        out[name] = a
        print(f'  [{label_str}] {name:>15s}: {a:.4f}')
    return out

print('A4 — homogeneous-K ensemble (A2 models):')
results['A4_aggregators_homo'] = aggregator_table(homo_logits, labels, 'A2-homo')
print('A4 — heterogeneous-K ensemble (A3 models):')
results['A4_aggregators_hetero'] = aggregator_table(hetero_logits, labels, 'A3-hetero')

# Entropy spread diagnostic
def entropy_spread(stacked_logits):
    p = F.softmax(stacked_logits, dim=-1)
    ent = -(p * p.clamp_min(1e-10).log()).sum(-1)
    return ent.std(0).mean().item()  # mean over batch of std across N

results['A4_entropy_spread_homo'] = entropy_spread(torch.stack(homo_logits))
results['A4_entropy_spread_hetero'] = entropy_spread(torch.stack(hetero_logits))
print(f'mean across-model entropy std: homo={results["A4_entropy_spread_homo"]:.4f}  '
      f'hetero={results["A4_entropy_spread_hetero"]:.4f}  '
      f'(higher = more diverse confidence between members)')
save_results()

## A5. Joint training (H5)

Same architecture as A3 (heterogeneous K), but trained end-to-end with the aggregated mean-of-probs NLL. Tests whether models specialize when given the chance.

**Collapse diagnostic:** measure pairwise prediction agreement rate of the jointly-trained models. If > 95%, they collapsed and the experiment is a wash.

In [ ]:
joint_models = []
for k, K in enumerate(K_HETERO):
    set_all_seeds(400 + k)
    joint_models.append(HashedDSCNN(K_per_layer=K, hash_seed=400 + k))
joint_models = train_joint(joint_models, epochs=EPOCHS_JOINT, log_prefix='[A5 joint]')

# Per-model accs and ensemble metrics under all aggregators
joint_logits = []; joint_single_accs = []
for k, m in enumerate(joint_models):
    a, lg, lbl = evaluate_logits(m, testloader)
    joint_logits.append(lg); joint_single_accs.append(a)
    print(f'A5 m{k}: single_acc={a:.4f}')

joint_aggs = aggregator_table(joint_logits, labels, 'A5-joint')

# Pairwise agreement diagnostic
preds_per_model = [lg.argmax(-1) for lg in joint_logits]
agree_pairs = []
for i in range(len(joint_models)):
    for j in range(i+1, len(joint_models)):
        agr = (preds_per_model[i] == preds_per_model[j]).float().mean().item()
        agree_pairs.append(agr)
mean_agree = float(np.mean(agree_pairs))
results['A5_joint'] = {
    'K_per_model': K_HETERO,
    'single_accs': joint_single_accs,
    'aggregators': joint_aggs,
    'mean_pairwise_agreement': mean_agree,
    'collapsed': mean_agree > 0.95,
}
print(f'A5 mean pairwise agreement: {mean_agree:.4f}  (> 0.95 = collapsed)')
save_results()

## A6. Layer-wise heterogeneous K (H7)

Three ensemble members, each with a **different layer-wise hashing profile** at the same total codebook budget:
- `uniform`: same K across all layers (the A2-style baseline)
- `pw_heavy`: hash only pointwise, leave dw/stem/fc dense — this isolates the conv-pw bottleneck
- `proportional`: K per layer ∝ √(virtual_params), via `make_proportional_K`

Tests whether architectural heterogeneity (where the hashing pressure lands) gives diversity beyond what same-K-different-seed can.

In [ ]:
# Build three different layer-wise K profiles. Anchor total per-model budget ≈ K_HOMO * (number of hashed layers).
sample_for_count = HashedDSCNN(K_per_layer=K_HOMO, hash_seed=0)
n_hashed_layers = sum(1 for m in sample_for_count.modules() if isinstance(m, HASHED_LAYER_TYPES))
target_total_per_model = K_HOMO * n_hashed_layers

# proportional profile (sqrt of virtual params)
prop_dict, layer_vsizes = make_proportional_K(target_total_per_model, sample_for_count)
del sample_for_count
print(f'Proportional K profile (target_total={target_total_per_model}, sum={sum(prop_dict.values())}):')
for name, K in prop_dict.items(): print(f'  {name:>30s}: K={K}')

# Map proportional dict back into HashedDSCNN's K_per_layer dict format
def prop_dict_to_kdict(d, num_blocks=NUM_BLOCKS):
    kd = {'stem': None, 'dw': [None]*num_blocks, 'pw': [None]*num_blocks, 'fc': None, 'se': None}
    se_Ks = []
    for name, K in d.items():
        if name == 'stem': kd['stem'] = K
        elif name == 'fc': kd['fc'] = K
        elif name.startswith('blocks.'):
            parts = name.split('.')
            block_idx = int(parts[1])
            sub = parts[2]   # 'dw', 'pw', or 'se'
            if sub == 'dw':
                kd['dw'][block_idx] = K
            elif sub == 'pw':
                kd['pw'][block_idx] = K
            elif sub == 'se':
                se_Ks.append(K)
    if se_Ks:
        kd['se'] = max(se_Ks)
    return kd

lw_configs = [
    {'name': 'uniform_K',     'kdict': K_HOMO},
    {'name': 'pw_heavy',      'kdict': {'stem': None, 'dw': [None]*NUM_BLOCKS,
                                         'pw': [target_total_per_model // NUM_BLOCKS]*NUM_BLOCKS,
                                         'fc': None, 'se': None}},
    {'name': 'proportional',  'kdict': prop_dict_to_kdict(prop_dict)},
]

lw_logits = []; lw_single_accs = []
for k, cfg in enumerate(lw_configs):
    set_all_seeds(500 + k)
    m = HashedDSCNN(K_per_layer=cfg['kdict'], hash_seed=500 + k)
    _, acc = train_one(m, log_prefix=f'[A6 {cfg["name"]}]')
    _, lg, lbl = evaluate_logits(m, testloader)
    lw_logits.append(lg); lw_single_accs.append(acc)
    print(f'A6 {cfg["name"]}: single_acc={acc:.4f}  real={m.real_params():,}')

lw_aggs = aggregator_table(lw_logits, labels, 'A6-layerwise')
results['A6_layerwise'] = {
    'configs': [c['name'] for c in lw_configs],
    'single_accs': lw_single_accs,
    'aggregators': lw_aggs,
    'proportional_dict': prop_dict,
}
save_results()

## A7. Dispersion test (H6)

For one config (`K = K_HOMO`), train `DISPERSION_N` models with different hash seeds. Measure:
- standard deviation of single-model test accuracy across seeds
- standard deviation of ensemble-of-pairs accuracy (resampled subsets) across seeds

**H6 prediction:** ensemble Std ≪ single-model Std. The robustness promise.

In [ ]:
disp_logits, disp_accs = [], []
for k in range(DISPERSION_N):
    set_all_seeds(600 + k)
    m = HashedDSCNN(K_per_layer=K_HOMO, hash_seed=600 + k)
    _, acc = train_one(m, log_prefix=f'[A7 m{k}]')
    _, lg, lbl = evaluate_logits(m, testloader)
    disp_logits.append(lg); disp_accs.append(acc)
    print(f'A7 m{k}: acc={acc:.4f}')

single_std = float(np.std(disp_accs))
single_mean = float(np.mean(disp_accs))

# Bootstrap-like: repeatedly sample N=3 of the trained models, take ensemble accuracy
rng = np.random.RandomState(0)
subset_accs = []
for _ in range(20):
    idx = rng.choice(DISPERSION_N, size=min(3, DISPERSION_N), replace=False)
    sub = [disp_logits[i] for i in idx]
    a = ensemble_accuracy(sub, labels, agg_mean_probs)
    subset_accs.append(a)
ens_std = float(np.std(subset_accs))
ens_mean = float(np.mean(subset_accs))

results['A7_dispersion'] = {
    'single_mean': single_mean, 'single_std': single_std, 'single_accs': disp_accs,
    'ensemble_subset_mean': ens_mean, 'ensemble_subset_std': ens_std,
    'subset_accs': subset_accs,
}
print(f'A7 single  mean={single_mean:.4f}  std={single_std:.4f}')
print(f'A7 N=3 ens mean={ens_mean:.4f}  std={ens_std:.4f}  (expect: ens_std much smaller)')
save_results()

## A8. OOD-lite robustness

Reuse trained models from A2 (homogeneous-K) and A3 (heterogeneous-K). Evaluate them on the corrupted CIFAR-10 (Gaussian noise + brightness shift). If the ensemble holds up better than the best single — and if the heterogeneous-K ensemble holds up better than homogeneous — that confirms the "different K = different failure modes" intuition under distribution shift.

In [ ]:
def collect_logits_on(loader, models):
    out = []
    for m in models:
        a, lg, lbl = evaluate_logits(m, loader)
        out.append((a, lg))
    return out, lbl

homo_ood, ood_labels = collect_logits_on(ood_loader, homo_models)
hetero_ood, _        = collect_logits_on(ood_loader, hetero_models)

homo_ood_singles = [a for a, _ in homo_ood]
hetero_ood_singles = [a for a, _ in hetero_ood]
homo_ood_ens   = ensemble_accuracy([lg for _, lg in homo_ood], ood_labels, agg_mean_probs)
hetero_ood_ens = ensemble_accuracy([lg for _, lg in hetero_ood], ood_labels, agg_mean_probs)

results['A8_ood'] = {
    'homo_singles': homo_ood_singles, 'homo_best': max(homo_ood_singles), 'homo_ens': homo_ood_ens,
    'hetero_singles': hetero_ood_singles, 'hetero_best': max(hetero_ood_singles), 'hetero_ens': hetero_ood_ens,
}
print(f'A8 OOD homo:   best_single={max(homo_ood_singles):.4f}  ensemble={homo_ood_ens:.4f}')
print(f'A8 OOD hetero: best_single={max(hetero_ood_singles):.4f}  ensemble={hetero_ood_ens:.4f}')
save_results()

## Summary table and headline plot

Pulls all the headline numbers into a single table. Most interesting comparisons:
- **H1**: A2 ensemble vs A1 single at `K = N·K_HOMO`
- **H2**: A3 ensemble vs A2 ensemble (matched total codebook budget)
- **H3-H4**: aggregator differences within A3 and A5
- **H5**: A5 joint ensemble vs A3 independent ensemble
- **H6**: A7 single std vs ensemble subset std
- **H7**: A6 layer-wise profile differences
- **A9**: branches=2 vs branches=1 at fixed K (does double hashing buy collision resilience inside a single model?)

In [ ]:
print('='*72)
print('SUMMARY')
print('='*72)
print(f'A0 Dense baseline:                            {results["A0_dense"]["acc"]:.4f}')
print('--- A1 single sweep ---')
for K, v in sorted(results['A1_sweep'].items()):
    print(f'  A1 single K={K:>6d}:                       {v["acc"]:.4f}')
print('--- A2 homogeneous-K ensemble ---')
print(f'  A2 best single (K={K_HOMO}):                  {results["A2_same_K"]["best_single"]:.4f}')
print(f'  A2 ensemble mean_probs:                      {results["A2_same_K"]["ensemble_meanp"]:.4f}')
print('--- A3 heterogeneous-K ensemble ---')
print(f'  A3 best single:                              {results["A3_different_K"]["best_single"]:.4f}')
print(f'  A3 ensemble mean_probs:                      {results["A3_different_K"]["ensemble_meanp"]:.4f}')
print('--- A4 aggregators on A3 ---')
for name, v in results['A4_aggregators_hetero'].items():
    print(f'  A4 hetero {name:>20s}:               {v:.4f}')
print('--- A5 joint training ---')
print(f'  A5 best single:                              {max(results["A5_joint"]["single_accs"]):.4f}')
print(f'  A5 ensemble mean_probs:                      {results["A5_joint"]["aggregators"]["mean_probs"]:.4f}')
print(f'  A5 mean pairwise agreement:                  {results["A5_joint"]["mean_pairwise_agreement"]:.4f}'
      f'  {"(COLLAPSED)" if results["A5_joint"]["collapsed"] else ""}')
print('--- A6 layer-wise ---')
for cfg, acc in zip(results['A6_layerwise']['configs'], results['A6_layerwise']['single_accs']):
    print(f'  A6 single {cfg:>20s}:               {acc:.4f}')
print(f'  A6 ensemble mean_probs:                      {results["A6_layerwise"]["aggregators"]["mean_probs"]:.4f}')
print('--- A7 dispersion ---')
print(f'  A7 single mean ± std:                        {results["A7_dispersion"]["single_mean"]:.4f} ± {results["A7_dispersion"]["single_std"]:.4f}')
print(f'  A7 N=3 ensemble subsets mean ± std:          {results["A7_dispersion"]["ensemble_subset_mean"]:.4f} ± {results["A7_dispersion"]["ensemble_subset_std"]:.4f}')
print('--- A8 OOD-lite ---')
print(f'  A8 homo best/ens:                            {results["A8_ood"]["homo_best"]:.4f} / {results["A8_ood"]["homo_ens"]:.4f}')
print(f'  A8 hetero best/ens:                          {results["A8_ood"]["hetero_best"]:.4f} / {results["A8_ood"]["hetero_ens"]:.4f}')
print('--- A9 parallel hash branches ---')
print(f'  A9 branches=1: mean ± std:                   {results["A9_branches"]["b1_mean"]:.4f} ± {results["A9_branches"]["b1_std"]:.4f}')
print(f'  A9 branches=2: mean ± std:                   {results["A9_branches"]["b2_mean"]:.4f} ± {results["A9_branches"]["b2_std"]:.4f}')
print(f'  A9 gain from double hashing:                 {results["A9_branches"]["gain"]:+.4f}')
print('='*72)
save_results()

## Summary table and headline plot

Pulls all the headline numbers into a single table. The most interesting comparisons:
- **H1**: A2 ensemble vs A1 at `K = N·K_HOMO`
- **H2**: A3 ensemble vs A2 ensemble (matched total codebook budget)
- **H3-H4**: aggregator differences within A3 and A5
- **H5**: A5 joint ensemble (best aggregator) vs A3 independent ensemble
- **H6**: A7 single std vs ensemble subset std
- **H7**: A6 layer-wise ensemble vs A2/A3 best ensemble


In [ ]:
print('='*70)
print('SUMMARY')
print('='*70)
print(f'A0 Dense baseline:                            {results["A0_dense"]["acc"]:.4f}')
print('--- A1 single sweep ---')
for K, v in sorted(results['A1_sweep'].items()):
    print(f'  A1 single K={K:>6d}:                       {v["acc"]:.4f}')
print('--- A2 homogeneous-K ensemble ---')
print(f'  A2 best single (K={K_HOMO}):                  {results["A2_same_K"]["best_single"]:.4f}')
print(f'  A2 ensemble mean_probs:                      {results["A2_same_K"]["ensemble_meanp"]:.4f}')
print('--- A3 heterogeneous-K ensemble ---')
print(f'  A3 best single:                              {results["A3_different_K"]["best_single"]:.4f}')
print(f'  A3 ensemble mean_probs:                      {results["A3_different_K"]["ensemble_meanp"]:.4f}')
print('--- A4 aggregators on A3 ---')
for name, v in results['A4_aggregators_hetero'].items():
    print(f'  A4 hetero {name:>20s}:               {v:.4f}')
print('--- A5 joint training ---')
print(f'  A5 best single:                              {max(results["A5_joint"]["single_accs"]):.4f}')
print(f'  A5 ensemble mean_probs:                      {results["A5_joint"]["aggregators"]["mean_probs"]:.4f}')
print(f'  A5 mean pairwise agreement:                  {results["A5_joint"]["mean_pairwise_agreement"]:.4f}'
      f'  {"(COLLAPSED)" if results["A5_joint"]["collapsed"] else ""}')
print('--- A6 layer-wise ---')
print(f'  A6 best single:                              {max(results["A6_layerwise"]["single_accs"]):.4f}')
print(f'  A6 ensemble mean_probs:                      {results["A6_layerwise"]["aggregators"]["mean_probs"]:.4f}')
print('--- A7 dispersion ---')
print(f'  A7 single mean ± std:                        {results["A7_dispersion"]["single_mean"]:.4f} ± {results["A7_dispersion"]["single_std"]:.4f}')
print(f'  A7 N=3 ensemble subsets mean ± std:          {results["A7_dispersion"]["ensemble_subset_mean"]:.4f} ± {results["A7_dispersion"]["ensemble_subset_std"]:.4f}')
print('--- A8 OOD-lite ---')
print(f'  A8 homo best/ens:                            {results["A8_ood"]["homo_best"]:.4f} / {results["A8_ood"]["homo_ens"]:.4f}')
print(f'  A8 hetero best/ens:                          {results["A8_ood"]["hetero_best"]:.4f} / {results["A8_ood"]["hetero_ens"]:.4f}')
print('='*70)
save_results()

In [ ]:
# Headline plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# left: A1 sweep + A2/A3 ensemble points
ks = sorted(results['A1_sweep'].keys())
accs = [results['A1_sweep'][k]['acc'] for k in ks]
axes[0].plot(ks, accs, 'o-', label='A1 single hashed')
axes[0].axhline(results['A0_dense']['acc'], color='k', linestyle='--', label='A0 dense')
axes[0].axhline(results['A2_same_K']['ensemble_meanp'], color='C1', linestyle=':',
                label=f'A2 ensemble homo K={K_HOMO}, N={ENSEMBLE_N}')
axes[0].axhline(results['A3_different_K']['ensemble_meanp'], color='C2', linestyle=':',
                label='A3 ensemble heterogeneous K')
axes[0].set_xscale('log'); axes[0].set_xlabel('K'); axes[0].set_ylabel('test acc')
axes[0].set_title('A1 sweep + ensemble references')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# right: aggregator comparison on heterogeneous ensemble
ag = results['A4_aggregators_hetero']
ag2 = results['A4_aggregators_homo']
names = list(ag.keys())
x = np.arange(len(names)); w = 0.4
axes[1].bar(x - w/2, [ag2[n] for n in names], w, label='homo (A2)')
axes[1].bar(x + w/2, [ag[n] for n in names], w, label='hetero (A3)')
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=20)
axes[1].set_ylabel('test acc')
axes[1].set_title('A4 — aggregator × ensemble type')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout(); plt.savefig('summary.png', dpi=100); plt.show()
print('Saved: summary.png, results.json')

## Next steps after running this notebook

1. **Paste the SUMMARY block into `NOTES.md` § 7** along with brief interpretive notes per hypothesis.
2. **Decide on H1 first.** If A2 ensemble does not beat A1 at matched total budget, the whole architectural angle is in trouble and we need to rethink.
3. **Decide on H2 next.** A3 vs A2 at matched total budget is the key signature observation. If A3 wins by ≥1-2%, heterogeneous K is real and worth formalizing further. If A3 ≈ A2, the diversity story needs a different mechanism.
4. **If H5 wins big** (joint > independent under same aggregator), look at per-class accuracy of each member to see if specialization is interpretable.
5. **Speech Commands follow-up.** If the CIFAR-10 results support the picture, repeat the key ablations on Speech Commands V2 — that's the actual KWS task and will land directly in the diploma.